<a href="https://colab.research.google.com/github/kuunalmistry/Google_Colab_Clg_Experiments_Projects/blob/main/Assignment_10_Transfer_Learning_Mobilenet_KuunalMistry_2410130.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install gradio --quiet

In [2]:
import os

In [3]:
import tensorflow as tf

In [4]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [5]:
from tensorflow.keras.applications import MobileNetV2

In [6]:
from tensorflow.keras.models import Sequential

In [7]:
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D

In [8]:
import gradio as gr

In [9]:
import numpy as np

In [10]:
from PIL import Image

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
dataset_path = "/content/drive/MyDrive/Brain_Tumor_Dataset"

In [4]:
# Parameters
img_size = 224
batch_size = 32

In [9]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Load dataset using ImageDataGenerator
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    horizontal_flip=True
)

In [11]:
train_data = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='binary',
    subset='training'
)

Found 164 images belonging to 2 classes.


In [12]:
val_data = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='binary',
    subset='validation'
)

Found 40 images belonging to 2 classes.


In [15]:
from tensorflow.keras.applications import MobileNetV2

# Transfer Learning using MobileNetV2
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(img_size, img_size, 3))
base_model.trainable = False # Freeze base

In [18]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(1, activation='sigmoid')
])

In [19]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=
['accuracy'])

In [20]:
history = model.fit(train_data, validation_data=val_data, epochs=5)

Epoch 1/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 113s 18s/step - accuracy: 0.3720 - loss: 0.9162 - val_accuracy: 0.6750 - val_loss: 0.6353
Epoch 2/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 8s 2s/step - accuracy: 0.7561 - loss: 0.5474 - val_accuracy: 0.8250 - val_loss: 0.4882
Epoch 3/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 8s 1s/step - accuracy: 0.8720 - loss: 0.3881 - val_accuracy: 0.8500 - val_loss: 0.3820
Epoch 4/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 9s 1s/step - accuracy: 0.9451 - loss: 0.2610 - val_accuracy: 0.8750 - val_loss: 0.3181
Epoch 5/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 9s 1s/step - accuracy: 0.9634 - loss: 0.1998 - val_accuracy: 0.9500 - val_loss: 0.2515


In [21]:
model.save("brain_tumor_model.h5")

In [25]:
import tensorflow as tf

model = tf.keras.models.load_model("brain_tumor_model.h5")
class_names = ['no', 'yes']

In [27]:
def predict_tumor(image):
  image = image.resize((img_size, img_size))
  img_array = tf.keras.utils.img_to_array(image) / 255.0
  img_array = np.expand_dims(img_array, axis=0)
  prediction = model.predict(img_array)[0][0]
  result = class_names[int(round(prediction))]
  confidence = prediction if result == 'yes' else 1 - prediction
  return f"Prediction: {result.upper()} ({confidence*100:.2f}% confidence)"

In [31]:
import gradio as gr

interface = gr.Interface(
    fn=predict_tumor,
    inputs=gr.Image(type="pil"),
    outputs="text",
    title="Brain Tumor Classifier",
    description="Upload a brain MRI image. The model predicts whether a tumor is present or not."
)

In [32]:
interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7eab55bc3cca5d4346.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
